## NeurIPS figures 5 — Spectral feedback sequences with ESMFold

This notebook loads **one sequence per target** from the held-out neurips eval CSVs (`feedbackmethod=spectral`), folds each with **ESMFold** (`esmfold_v1`) following `DRAKES/drakes_protein/fmif/post_processing_scripts/scrmsd.py` (invoked by `DRAKES/drakes_protein/fmif/post_processing_scripts/eval_scrmsd.sh`), optionally applies the same **PyRosetta FastRelax** as that pipeline, and visualizes **native vs. predicted** structures.

**Naming:** eval files use `r6_560_TrROS_Hall.pdb`. There is no `r6_650` entry in this repo; figures use the display label **r6_650_TrROS_Hall** while resolving paths from `r6_560_TrROS_Hall.pdb`.

**Runtime (order of magnitude):** First-time **ESMFold** checkpoint download/load can take **several minutes**. After that, on a **GPU**, each ~50–60 residue fold is often **~30 s–3 min**; on **CPU only**, folding can be **tens of minutes per protein** or fail depending on the environment—**a CUDA GPU is strongly recommended**. **PyRosetta FastRelax** (native + predicted, when installed) is often **~1–5+ minutes per structure**. All-in for **two proteins**, expect roughly **~5–25 min with GPU + relax**, or much longer without a GPU.

**PyMOL (optional, publication-quality):** The CLI script `neurips_figures5.py` also writes **ray-traced PNGs** (`*_esmfold_pymol.png` next to the PDFs): silver cartoon for the native structure and **blue–white–red by B-factor** for the prediction (ESMFold pLDDT). Requires `pymol` on `PATH` (e.g. `conda install -c conda-forge pymol-open-source`). Use `--no-pymol` to skip.

After the folding cells below have built `results`, you can run the **optional cell** at the end of this notebook to render PyMOL figures without re-running the script.

In [1]:
# Optional: pip install py3dmol  (for interactive 3D in Jupyter)
import os
import shutil
import tempfile

import numpy as np
import pandas as pd
import torch
import esm

REPO = "/home/shai/BLISS_Experiments/DRAKES"
# PDFs written here (vector CA-backbone overlay)
FIG_PDF_DIR = os.path.join(REPO, "neurips_fig5_pdfs")
FMIF = os.path.join(REPO, "DRAKES/drakes_protein/fmif")
# Same run family as other neurips figures; change masks=... if you use a different budget.
EVAL_CSV = os.path.join(
    FMIF,
    "eval_results/neurips/pretrained_test_ddg_bon_N=1_feedbacksteps=1_feedbackmethod=spectral_maxspecorder=20_masks=4096_rmax=False_.csv",
)
PDB_DIR = os.path.join(
    REPO,
    "DRAKES/data/data_and_model/proteindpo_data/AlphaFold_model_PDBs",
)
# Mirrors folding_model.FoldingModel + scrmsd.py (torch.hub cache)
PT_HUB = os.path.join(REPO, "DRAKES/data/data_and_model/.cache/torch/")

# (protein_name column in CSV, display title for figures)
TARGETS = [
    ("r6_560_TrROS_Hall.pdb", "r6_650_TrROS_Hall"),
    ("2KRU.pdb", "2KRU"),
]

In [2]:
df = pd.read_csv(EVAL_CSV)

selected = {}
for pdb_fn, title in TARGETS:
    sub = df[df["protein_name"] == pdb_fn]
    if sub.empty:
        raise FileNotFoundError(f"No rows with protein_name={pdb_fn!r} in {EVAL_CSV}")
    # First spectral-feedback row per protein (swap for e.g. sub.loc[sub["scrmsd"].idxmin()] to pick lowest scRMSD)
    row = sub.iloc[0]
    selected[pdb_fn] = row
    scr = row["scrmsd"] if "scrmsd" in row.index else None
    print(f"{title}: n={len(row['seq'])}  scrmsd={scr}")

In [3]:
def fold_sequence_esmfold(seq: str, out_pdb: str, device: str, model):
    """Match `multiflow.models.folding_model.FoldingModel._esmf_model` (single chain)."""
    seq = seq.replace("X", "A")
    if model is None:
        os.makedirs(PT_HUB, exist_ok=True)
        torch.hub.set_dir(PT_HUB)
        _dev = torch.device(device)
        model = esm.pretrained.esmfold_v1().eval().to(_dev)
        # FP16 LayerNorm is not implemented on CPU in PyTorch; force FP32.
        if _dev.type == "cpu":
            model = model.float()
    with torch.no_grad():
        out = model.infer(seq)
        pdb_str = model.output_to_pdb(out)[0]
    with open(out_pdb, "w") as f:
        f.write(pdb_str)
    plddt = out["mean_plddt"][0].item()
    return model, plddt


_PYRO_INIT = False


def relax_pdb_like_scrmsd(pdb_path: str) -> bool:
    """Match `post_processing_scripts/scrmsd.py` `relax_pose` when PyRosetta is available."""
    global _PYRO_INIT
    try:
        import pyrosetta
        from pyrosetta.rosetta.core.pack.task import TaskFactory
        from pyrosetta.rosetta.protocols.minimization_packing import PackRotamersMover
        from pyrosetta.rosetta.protocols.relax import FastRelax
        from pyrosetta.rosetta.core.pack.task.operation import RestrictToRepacking
    except ImportError:
        return False
    if not _PYRO_INIT:
        pyrosetta.init(extra_options="-out:level 100")
        _PYRO_INIT = True
    scorefxn = pyrosetta.create_score_function("ref2015_cart")
    pose = pyrosetta.pose_from_file(pdb_path)
    tf = TaskFactory()
    tf.push_back(RestrictToRepacking())
    packer = PackRotamersMover(scorefxn, tf.create_task_and_apply_taskoperations(pose))
    packer.apply(pose)
    relax = FastRelax()
    relax.set_scorefxn(scorefxn)
    relax.apply(pose)
    pose.dump_pdb(pdb_path)
    return True

In [4]:
import biotite.structure as struc
from biotite.structure.io.pdb import PDBFile


def align_prediction_to_native(native_pdb: str, pred_pdb: str, out_pdb: str) -> None:
    """Superimpose predicted structure onto native using CA atoms; write aligned PDB."""
    nat = PDBFile.read(native_pdb).get_structure(model=1)
    pred = PDBFile.read(pred_pdb).get_structure(model=1)
    nat_ca = nat[nat.atom_name == "CA"]
    pred_ca = pred[pred.atom_name == "CA"]
    L = min(len(nat_ca), len(pred_ca))
    if len(nat_ca) != len(pred_ca):
        print(f"  CA count mismatch native={len(nat_ca)} pred={len(pred_ca)} — using first {L} residues")
    nat_ca = nat_ca[:L]
    pred_ca = pred_ca[:L]
    _, trafo = struc.superimpose(nat_ca, pred_ca)
    pred_copy = pred.copy()
    # Older biotite: trafo is (t1, rot, t2); newer: AffineTransformation with .apply()
    if isinstance(trafo, tuple):
        from biotite.structure import superimpose_apply as _sup_apply
        pred_aligned = _sup_apply(pred_copy, trafo)
    else:
        pred_aligned = trafo.apply(pred_copy)
    file = PDBFile()
    file.set_structure(pred_aligned)
    file.write(out_pdb)


def show_native_vs_pred(native_pdb: str, pred_aligned_pdb: str, title: str):
    try:
        import py3Dmol
    except ImportError:
        print("Install py3Dmol for 3D views: pip install py3dmol")
        print(" native:", native_pdb, "\n pred:  ", pred_aligned_pdb)
        return
    with open(native_pdb) as f:
        nat = f.read()
    with open(pred_aligned_pdb) as f:
        pred = f.read()
    view = py3Dmol.view(width=700, height=480)
    view.addModel(nat, "pdb")
    view.setStyle({"model": 0}, {"cartoon": {"color": "#bbbbbb"}})
    view.addModel(pred, "pdb")
    view.setStyle({"model": 1}, {"cartoon": {"color": "#2ecc71"}})
    view.zoomTo()
    print(title)
    view.show()


def _ca_coords(pdb_path: str):
    atoms = PDBFile.read(pdb_path).get_structure(model=1)
    ca = atoms[atoms.atom_name == "CA"]
    return ca.coord.copy()


def save_ca_overlay_pdf(
    native_pdb: str,
    pred_aligned_pdb: str,
    pdf_path: str,
    title: str,
    dpi: int = 150,
) -> None:
    """Save a vector PDF: CA backbone native vs aligned prediction (matplotlib 3D)."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

    nat = _ca_coords(native_pdb)
    pred = _ca_coords(pred_aligned_pdb)
    L = min(len(nat), len(pred))
    nat = nat[:L]
    pred = pred[:L]
    pts = np.vstack([nat, pred])

    fig = plt.figure(figsize=(5.5, 5))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot(nat[:, 0], nat[:, 1], nat[:, 2], color="#888888", linewidth=1.4, label="Native")
    ax.plot(pred[:, 0], pred[:, 1], pred[:, 2], color="#27ae60", linewidth=1.4, label="ESMFold (spectral seq.)")

    cr = pts.min(axis=0)
    cl = pts.max(axis=0)
    ctr = (cr + cl) / 2
    span = (cl - cr).max() / 2 + 1e-6
    ax.set_xlim(ctr[0] - span, ctr[0] + span)
    ax.set_ylim(ctr[1] - span, ctr[1] + span)
    ax.set_zlim(ctr[2] - span, ctr[2] + span)
    try:
        ax.set_box_aspect((1, 1, 1))
    except AttributeError:
        pass

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.legend(loc="upper right", fontsize=8)
    fig.suptitle(title, fontsize=10)
    os.makedirs(os.path.dirname(os.path.abspath(pdf_path)) or ".", exist_ok=True)
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight", dpi=dpi)
    plt.close(fig)
    print(f"Saved PDF: {pdf_path}")

In [5]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("device:", device)

workdir = tempfile.mkdtemp(prefix="neurips_fig5_")
print("working dir:", workdir)

esm_model = None
results = {}
for pdb_fn, title in TARGETS:
    seq = selected[pdb_fn]["seq"]
    raw_pred = os.path.join(workdir, f"raw_{pdb_fn.replace('.pdb', '')}.pdb")
    aln_pred = os.path.join(workdir, f"aligned_{pdb_fn.replace('.pdb', '')}.pdb")
    native_src = os.path.join(PDB_DIR, pdb_fn)
    # `scrmsd.py` also relaxes the reference pose (`get_true_pose`); mirror that for the gray model
    native_for_align = os.path.join(workdir, f"native_relaxed_{pdb_fn.replace('.pdb', '')}.pdb")
    shutil.copy(native_src, native_for_align)
    _ = relax_pdb_like_scrmsd(native_for_align)

    esm_model, plddt = fold_sequence_esmfold(seq, raw_pred, device, esm_model)
    ok = relax_pdb_like_scrmsd(raw_pred)
    if not ok:
        print("  (PyRosetta relax skipped for prediction — environment without PyRosetta)")

    align_prediction_to_native(native_for_align, raw_pred, aln_pred)
    results[pdb_fn] = {
        "title": title,
        "seq": seq,
        "plddt": plddt,
        "native_pdb": native_for_align,
        "pred_pdb_raw": raw_pred,
        "pred_pdb_aligned": aln_pred,
    }
    print(f"{title}: mean pLDDT ≈ {plddt:.2f}")

In [ ]:
os.makedirs(FIG_PDF_DIR, exist_ok=True)

for pdb_fn, meta in results.items():
    caption = (
        f"{meta['title']} — native vs ESMFold (spectral seq.), "
        f"mean pLDDT={meta['plddt']:.2f}"
    )
    safe_name = meta["title"].replace(" ", "_").replace("/", "-")
    pdf_path = os.path.join(FIG_PDF_DIR, f"{safe_name}_esmfold_overlay.pdf")
    save_ca_overlay_pdf(
        meta["native_pdb"],
        meta["pred_pdb_aligned"],
        pdf_path,
        title=caption,
    )
    show_native_vs_pred(
        meta["native_pdb"],
        meta["pred_pdb_aligned"],
        title=f"{meta['title']} — native (gray) vs ESMFold spectral sequence (green); pLDDT={meta['plddt']:.2f}",
    )

In [ ]:
# Optional: PyMOL ray-traced PNGs (requires `pymol` on PATH — conda-forge pymol-open-source)
import importlib.util
from pathlib import Path

_script = Path(REPO) / "neurips_figures5.py"
if not _script.is_file():
    raise FileNotFoundError(_script)

spec = importlib.util.spec_from_file_location("neurips_figures5", _script)
_m = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(_m)

for _fn, meta in results.items():
    safe = meta["title"].replace(" ", "_").replace("/", "-")
    out_png = Path(FIG_PDF_DIR) / f"{safe}_esmfold_pymol.png"
    ok = _m.render_pymol_overlay_png(
        meta["native_pdb"],
        meta["pred_pdb_aligned"],
        str(out_png),
        ray_w=3200,
        ray_h=3200,
    )
    if not ok:
        break